In [7]:
import random
from dataclasses import dataclass
from enum import Enum

from deap import base, creator, tools

In [8]:
class SensorType(Enum):
    LIDAR_16_CH = 1     # e.g., 16-channel spinning LiDAR
    LIDAR_32_CH = 2     # e.g., 32-channel spinning LiDAR
    SOLID_STATE = 3     # e.g., Directional solid-state LiDAR


@dataclass
class Gene:
    sensor_type: SensorType
    node_id: int
    pitch: int
    roll: int


VALID_NODE_IDS = list(range(0, 200))

In [ ]:
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

In [10]:
def create_gene(assigned_node_id):
    """Generates a single active sensor configuration."""
    sensor_type = random.choice(list(SensorType))

    # Define physical rotation bounds.
    # Note: If you are using 360-degree spinning LiDARs, roll/pitch is sufficient.
    # If using directional solid-state LiDARs, you may need to swap 'roll' for 'yaw'.
    pitch = random.randint(-90, 90)
    roll = random.randint(-180, 180)

    return Gene(sensor_type=sensor_type, node_id=assigned_node_id, pitch=pitch, roll=roll)


def create_individual(icls):
    """Generates an individual with 1 to 4 active sensors."""
    num_sensors = random.randint(1, 4)

    # Ensure unique node IDs for each sensor in the individual
    unique_nodes = random.sample(VALID_NODE_IDS, num_sensors)

    genes = [create_gene(node_id) for node_id in unique_nodes]

    # Initialize the DEAP list class with our generated genes
    return icls(genes)

In [13]:
toolbox = base.Toolbox()

# Register the individual generator
toolbox.register("individual", create_individual, creator.Individual)

# Register the population generator (creates a list of individuals)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Create initial population
population = toolbox.population(n=100)

In [28]:
from IPython.display import HTML, display


def visualize_population(population):
    sensor_colors = {
        SensorType.LIDAR_16_CH: "#1f77b4",
        SensorType.LIDAR_32_CH: "#ff7f0e",
        SensorType.SOLID_STATE: "#2ca02c",
    }

    max_genes = max(len(individual) for individual in population)
    rows = []

    for index, individual in enumerate(population, start=1):
        cells = [f'<th>Individual {index}</th>']

        for gene in individual:
            color = sensor_colors[gene.sensor_type]
            cells.append(
                f'<td style="border-left: 6px solid {color}; background: {color}22;">'
                f'<div style="font-weight: 700; margin-bottom: 4px;">{gene.sensor_type.name}</div>'
                f'<div>node {gene.node_id}</div>'
                f'<div>pitch {gene.pitch}, roll {gene.roll}</div>'
                f'</td>'
            )

        while len(cells) < max_genes + 1:
            cells.append('<td class="empty">&mdash;</td>')

        rows.append('<tr>' + ''.join(cells) + '</tr>')

    header = '<tr><th>Individual</th>' + ''.join(f'<th>Sensor {i}</th>' for i in range(1, max_genes + 1)) + '</tr>'
    html = f'''
    <style>
      .population-table {{
        border-collapse: collapse;
        width: 100%;
        font-family: Arial, sans-serif;
        font-size: 14px;
      }}
      .population-table th, .population-table td {{
        border: 1px solid #d0d7de;
        padding: 10px 12px;
        vertical-align: top;
      }}
      .population-table th {{
        background: #f6f8fa;
        text-align: center;
      }}
      .population-table .empty {{
        color: #8c959f;
        text-align: center;
      }}
    </style>
    <table class='population-table'>
      {header}
      {''.join(rows)}
    </table>
    '''

    display(HTML(html))

population = toolbox.population(n=100)
visualize_population(population)

Individual,Sensor 1,Sensor 2,Sensor 3,Sensor 4
Individual 1,"SOLID_STATEnode 132pitch 2, roll 86","SOLID_STATEnode 22pitch 31, roll 111","LIDAR_32_CHnode 62pitch 27, roll 125",—
Individual 2,"LIDAR_16_CHnode 116pitch 81, roll 92",—,—,—
Individual 3,"LIDAR_32_CHnode 167pitch 23, roll -164","LIDAR_16_CHnode 11pitch 18, roll -141","SOLID_STATEnode 157pitch 7, roll 35","LIDAR_32_CHnode 0pitch -83, roll 59"
Individual 4,"LIDAR_32_CHnode 98pitch -65, roll -176",—,—,—
Individual 5,"SOLID_STATEnode 196pitch 3, roll -60","LIDAR_16_CHnode 191pitch 10, roll 52","SOLID_STATEnode 20pitch 86, roll -117",—
Individual 6,"LIDAR_32_CHnode 4pitch -15, roll 83","LIDAR_32_CHnode 184pitch 42, roll -17","SOLID_STATEnode 112pitch -39, roll 2",—
Individual 7,"LIDAR_32_CHnode 181pitch -10, roll -80",—,—,—
Individual 8,"LIDAR_16_CHnode 55pitch -59, roll -77",—,—,—
Individual 9,"SOLID_STATEnode 106pitch 19, roll -162","LIDAR_16_CHnode 6pitch -53, roll -111",—,—
Individual 10,"LIDAR_32_CHnode 149pitch 66, roll 19",—,—,—
